# Bronze -> Silver: visão consolidada

ADR [0022](../../docs/adr/0022-notebooks-de-diagnostico-medallion-separados-da-narrativa-do-tcc.md)
(issue [#128](https://github.com/Vini0606/Tecnicas-de-Ciencia-de-Dados-em-dados-do-Instagram/issues/128)).
Volumetria, completude e o que a limpeza Silver muda em relação à Bronze, para as 3 tabelas Bronze +
5 Silver de uma vez -- estes dois estágios são ingestão/limpeza, não modelagem, então não têm
profundidade individual suficiente para justificar um notebook por tabela (ver Gold em
`gold_*.ipynb`, um por domínio de modelagem).

Notebook estritamente leitura: nunca chama `run_medallion_pipeline` nem nenhum `*Cleaner.write` --
mesmo princípio da ADR [0003](../../docs/adr/0003-desacoplar-modelagem-do-notebook-via-scripts-cli-com-checkpoint.md).

In [ ]:
import sys
import os

# Adiciona o diretório raiz do projeto ao sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
from deltalake import DeltaTable
from dotenv import load_dotenv

from config import settings
from src.data_extract.bronze_writer import BronzeWriter
from src.repositories.delta_repository import DeltaRepository
from src.analysis.medallion_diagnostics import completeness_summary, count_duplicate_rows

load_dotenv()
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)

## Carga

Bronze via `BronzeWriter.get_latest_*` -- o mesmo acesso que `pipeline.py` usa para checar se já há
dado local (`_bronze_has_data`). `DeltaRepository` cobre `posts_clean`/`reels_clean`/
`governors_metadata`, mas não expõe `profiles_clean`/`comments_clean` diretamente (`load_profiles()`
do repositório na verdade lê `governor_engagement`, já em Gold -- nome legado do repositório, não
confundir); essas duas leem a tabela Delta diretamente pelo path de `settings`, mesmo primitivo que
`DeltaRepository._load` usa por baixo.

In [ ]:
bronze = BronzeWriter(
    bronze_profiles_path=settings.BRONZE_PROFILES,
    bronze_posts_path=settings.BRONZE_POSTS,
    bronze_reels_path=settings.BRONZE_REELS,
)
repo = DeltaRepository(gold_dir=settings.GOLD_DIR, silver_dir=settings.SILVER_DIR)

tabelas = {
    'bronze.instagram_profiles': bronze.get_latest_profiles(),
    'bronze.instagram_posts': bronze.get_latest_posts(),
    'bronze.instagram_reels': bronze.get_latest_reels(),
    'silver.profiles_clean': DeltaTable(str(settings.SILVER_PROFILES)).to_pandas(),
    'silver.posts_clean': repo.load_posts(),
    'silver.reels_clean': repo.load_reels(),
    'silver.comments_clean': DeltaTable(str(settings.SILVER_COMMENTS)).to_pandas(),
    'silver.governors_metadata': repo.load_governors_metadata(),
}

## Volumetria e duplicatas

In [ ]:
resumo = pd.DataFrame({
    'tabela': list(tabelas.keys()),
    'n_linhas': [len(df) for df in tabelas.values()],
    'n_colunas': [df.shape[1] for df in tabelas.values()],
    'n_linhas_duplicadas': [count_duplicate_rows(df) for df in tabelas.values()],
})
resumo

## Completude por tabela

Só colunas com pelo menos um nulo aparecem -- uma tabela sem saída abaixo do seu nome está
100% completa nas colunas que tem.

In [ ]:
for nome, df in tabelas.items():
    resumo_completude = completeness_summary(df)
    colunas_com_nulo = resumo_completude[resumo_completude['n_nulos'] > 0]
    if not colunas_com_nulo.empty:
        print(f'--- {nome} ---')
        display(colunas_com_nulo)

## O que a limpeza Silver muda em relação à Bronze

Para os três pares com correspondência direta (`profiles`, `posts`, `reels`): diferença de colunas
(o que a limpeza remove/deriva) e diferença de contagem de linhas (o que é descartado, ex.: perfis
sem dado suficiente para conformar o schema Silver).

In [ ]:
pares_bronze_silver = {
    'profiles': ('bronze.instagram_profiles', 'silver.profiles_clean'),
    'posts': ('bronze.instagram_posts', 'silver.posts_clean'),
    'reels': ('bronze.instagram_reels', 'silver.reels_clean'),
}

linhas_comparacao = []
for entidade, (nome_bronze, nome_silver) in pares_bronze_silver.items():
    df_bronze = tabelas[nome_bronze]
    df_silver = tabelas[nome_silver]
    linhas_comparacao.append({
        'entidade': entidade,
        'linhas_bronze': len(df_bronze),
        'linhas_silver': len(df_silver),
        'linhas_descartadas': len(df_bronze) - len(df_silver),
        'colunas_removidas_pela_limpeza': sorted(set(df_bronze.columns) - set(df_silver.columns)),
        'colunas_derivadas_pela_limpeza': sorted(set(df_silver.columns) - set(df_bronze.columns)),
    })

pd.DataFrame(linhas_comparacao)

## Nota de interpretação

Comparar `linhas_descartadas` acima com o motivo de descarte esperado (perfil/post sem os campos
mínimos que `ProfileCleaner`/`PostCleaner` exigem -- ver `src/features/silver/`) antes de assumir que
uma queda grande é um bug de coleta: para um universo de ~27 governadores, descartar 1-3 linhas por
tabela costuma ser esperado (perfis novos sem posts suficientes, ou posts com campo crítico ausente
vindos do scraper), não um sinal de problema sistemático. Se `linhas_descartadas` crescer muito acima
disso numa execução futura, vale investigar a extração (Bronze) antes de suspeitar da limpeza
(Silver).